# Algorithm Benchmark

This notebook compares RSA, ML-KEM, ML-DSA, SLH-DSA, and HQC for the payment transaction security use case.

Implementation note: this notebook does not duplicate benchmark logic. It imports and visualizes functions from `benchmarks.algorithm_benchmark`, so the package module remains the single source of truth.

Notes:
- ML-KEM and HQC are KEM/encryption primitives for protecting payment data keys.
- ML-DSA and SLH-DSA are signature primitives for transaction authorization.
- If `liboqs` or `cryptography` is unavailable, the benchmark module clearly reports `simulation fallback`.

In [1]:
from pathlib import Path
import importlib
import sys

project_root = Path.cwd()
if (project_root / "src").exists():
    sys.path.insert(0, str(project_root / "src"))
else:
    sys.path.insert(0, str(project_root))

from algorithms.crypto_algorithms import list_algorithm_profiles
import benchmarks.algorithm_benchmark as benchmark_module
benchmark_module = importlib.reload(benchmark_module)

attack_estimates_as_dicts = benchmark_module.attack_estimates_as_dicts
benchmark_results_as_dicts = benchmark_module.benchmark_results_as_dicts
toy_crack_results_as_dicts = benchmark_module.toy_crack_results_as_dicts
CRYPTOGRAPHY_AVAILABLE = benchmark_module.CRYPTOGRAPHY_AVAILABLE
OQS_AVAILABLE = benchmark_module.OQS_AVAILABLE

print(f"cryptography available: {CRYPTOGRAPHY_AVAILABLE}")
print(f"liboqs available: {OQS_AVAILABLE}")

cryptography available: True
liboqs available: False


## Security Scorecard

The base score is the starting cryptographic posture score before transaction-specific penalties are applied.

In [2]:
try:
    import pandas as pd
    profiles_df = pd.DataFrame(list_algorithm_profiles())
    display(profiles_df[[
        "name",
        "primary_use",
        "nist_status",
        "quantum_safe",
        "security_level",
        "base_score",
        "production_role",
    ]])
except ImportError:
    for profile in list_algorithm_profiles():
        print(profile)

,name,primary_use,nist_status,quantum_safe,security_level,base_score,production_role
0,RSA-2048,Encryption/signature,Classical legacy,False,Classical ~112-bit; vulnerable to Shor's algor...,46,Legacy baseline for comparison
1,ML-KEM-768,Key encapsulation/encryption,FIPS 203,True,NIST category 3,93,Primary PQC encryption/KEM choice
2,ML-DSA-65,Digital signature,FIPS 204,True,NIST category 3,91,Primary PQC signature choice
3,SLH-DSA-128s,Digital signature,FIPS 205,True,NIST category 1,84,Conservative backup signature option
4,HQC-128,Key encapsulation/encryption,Selected by NIST in 2025 for future standardiz...,True,NIST category 1 target,82,Backup KEM for crypto-agility


## Run Local Benchmark

Increase `iterations` for a more stable comparison. Keep it small when running on an older laptop or without native wheels installed.

In [3]:
iterations = 5
benchmark_rows = benchmark_results_as_dicts(iterations=iterations)

try:
    benchmark_df = pd.DataFrame(benchmark_rows)
    display(benchmark_df)
except NameError:
    for row in benchmark_rows:
        print(row)

,algorithm,primitive,implementation,avg_ms,p95_ms,security_score,attack_time_score,estimated_classical_attack_years,estimated_quantum_attack_years,attack_model,quantum_safe,notes
0,RSA-2048,Encryption + signature,cryptography real RSA,83.161,149.463,46,35,~1e19.5 years,10.00 years,"RSA-2048 is classically hard, but Shor's algor...",False,"Legacy baseline: real RSA-OAEP key wrap, AES-G..."
1,ML-KEM-768,KEM + AEAD encryption,simulation fallback,0.060,0.270,93,100,~1e43.6 years,~1e24.3 years,NIST category 3 lattice KEM; estimate uses bru...,True,Primary PQC encryption/KEM path for protecting...
2,ML-DSA-65,Digital signature,simulation fallback,0.086,0.094,91,100,~1e43.6 years,~1e24.3 years,NIST category 3 lattice signature; estimate us...,True,Primary PQC transaction authorization signature.
3,SLH-DSA-128s,Digital signature,simulation fallback,0.165,0.169,84,75,~1e24.3 years,~1e5.0 years,Hash-based signature; quantum estimate reflect...,True,Hash-based backup signature; useful for crypto...
4,HQC-128,KEM + AEAD encryption,simulation fallback,0.009,0.011,82,75,~1e24.3 years,~1e5.0 years,Code-based KEM candidate; estimate uses catego...,True,Code-based backup KEM candidate with different...


## Toy Crack Demo

This is the actual crack-time demo. It uses the same payment transaction payload for every algorithm label, then brute-forces intentionally tiny demo key spaces. The result is useful for observing relative crack-time growth, but it is **not** a real crack of RSA-2048, ML-KEM, ML-DSA, SLH-DSA, or HQC.

In [4]:
toy_crack_rows = toy_crack_results_as_dicts()

try:
    toy_crack_df = pd.DataFrame(toy_crack_rows)
    display(toy_crack_df[[
        "algorithm",
        "toy_key_bits",
        "toy_keyspace",
        "crack_time_ms",
        "toy_crack_score",
        "notes",
    ]])
except NameError:
    for row in toy_crack_rows:
        print(row)

,algorithm,toy_key_bits,toy_keyspace,crack_time_ms,toy_crack_score,notes
0,RSA-2048,12,4096,3.884,25,Actual brute-force crack of a tiny demo keyspa...
1,HQC-128,14,16384,9.941,25,Actual brute-force crack of a tiny demo keyspa...
2,SLH-DSA-128s,14,16384,18.317,25,Actual brute-force crack of a tiny demo keyspa...
3,ML-KEM-768,15,32768,30.588,45,Actual brute-force crack of a tiny demo keyspa...
4,ML-DSA-65,15,32768,26.481,45,Actual brute-force crack of a tiny demo keyspa...


## Attack-Time Estimate

This section adds a crack-time-oriented score. It does **not** actually crack RSA-2048 or PQC keys. Instead, it calibrates a local trial-operation rate and estimates attack time from brute-force-equivalent security strength. This makes attack resistance a separate scoring option beside deployment/security posture.

In [5]:
attack_rows = attack_estimates_as_dicts()

try:
    attack_df = pd.DataFrame(attack_rows)
    display(attack_df[[
        "algorithm",
        "estimated_classical_attack_years",
        "estimated_quantum_attack_years",
        "attack_time_score",
        "attack_model",
    ]])
except NameError:
    for row in attack_rows:
        print(row)

,algorithm,estimated_classical_attack_years,estimated_quantum_attack_years,attack_time_score,attack_model
0,RSA-2048,~1e19.5 years,10.00 years,35,"RSA-2048 is classically hard, but Shor's algor..."
1,ML-KEM-768,~1e43.6 years,~1e24.3 years,100,NIST category 3 lattice KEM; estimate uses bru...
2,ML-DSA-65,~1e43.6 years,~1e24.3 years,100,NIST category 3 lattice signature; estimate us...
3,SLH-DSA-128s,~1e24.3 years,~1e5.0 years,75,Hash-based signature; quantum estimate reflect...
4,HQC-128,~1e24.3 years,~1e5.0 years,75,Code-based KEM candidate; estimate uses catego...


## Payment Security Interpretation

Use **ML-KEM-768** to protect payment payload data keys and **ML-DSA-65** to sign transaction authorization messages. Keep **SLH-DSA** and **HQC** as backup families for crypto-agility.